# Fine-tune Gemma 2 2B-it for C/C++ vulnerability detection

This Colab notebook trains a QLoRA adapter on prepared Devign JSONL files. Use a free T4 runtime, and store inputs/outputs on Google Drive because Colab disk is ephemeral.

In [ ]:
# If this fails after a Colab update, check Unsloth's official Colab notebook for its current compatible pins.
!pip -q install 'unsloth[colab-new]' 'transformers>=4.45,<4.49' 'peft>=0.12,<0.14' 'trl>=0.11,<0.13' 'bitsandbytes>=0.43' 'accelerate>=0.33' datasets

## 1. Mount Drive and configure the experiment

Before running, put `train.jsonl`, `validation.jsonl`, and `test.jsonl` in `MyDrive/slm-vuln-detection/data/processed/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Keep these values in sync with src/config.py.
MODEL_NAME = 'google/gemma-2-2b-it'
MAX_SEQ_LENGTH = 2048
LORA_RANK, LORA_ALPHA = 16, 32
BATCH_SIZE, GRAD_ACCUM, LEARNING_RATE, EPOCHS = 2, 4, 2e-4, 2
DRIVE_ROOT = '/content/drive/MyDrive/slm-vuln-detection'
DATA_DIR, OUTPUT_DIR = f'{DRIVE_ROOT}/data/processed', f'{DRIVE_ROOT}/models/gemma2-devign-lora'
PREDICTIONS_FILE = f'{DRIVE_ROOT}/results/model_predictions.jsonl'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(os.path.dirname(PREDICTIONS_FILE), exist_ok=True)

## 2. Shared prompt contract

**SYNC MARKER:** This cell is the intentional Colab copy of `src/dataset_utils.py`. Update both together: training and inference must use this exact template.

In [ ]:
VULNERABLE, NOT_VULNERABLE = 'Vulnerable', 'Not vulnerable'
PROMPT_TEMPLATE = '''You are a software security reviewer. Analyze the following C/C++ function for a security vulnerability. Reply with exactly one label: Vulnerable or Not vulnerable.\n\nCode:\n```c\n{code}\n```\n\nAnswer:'''
def build_prompt(code): return PROMPT_TEMPLATE.format(code=code.strip())
def training_text(row): return f"{build_prompt(row['code'])} {row['response']}"
def parse_binary_prediction(text):
    first = text.strip().splitlines()[0].lower() if text.strip() else ''
    if 'not vulnerable' in first or 'not_vulnerable' in first: return NOT_VULNERABLE
    return VULNERABLE if 'vulnerable' in first else NOT_VULNERABLE

## 3. Load Gemma in 4-bit and attach LoRA

You must have accepted the Gemma license on Hugging Face and be authenticated in Colab if prompted.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth', random_state=42)

## 4. Load prepared splits and train

The JSONL schema was produced by `data/prepare_dataset.py`; no local GPU is required for its preparation.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
files = {'train': f'{DATA_DIR}/train.jsonl', 'validation': f'{DATA_DIR}/validation.jsonl', 'test': f'{DATA_DIR}/test.jsonl'}
ds = load_dataset('json', data_files=files)
def add_text(batch): return {'text': [training_text({'code': c, 'response': r}) for c, r in zip(batch['code'], batch['response'])]}
train_ds = ds['train'].map(add_text, batched=True)
val_ds = ds['validation'].map(add_text, batched=True)
args = TrainingArguments(output_dir='/content/gemma2_checkpoints', per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS, logging_steps=10,
    eval_strategy='steps', eval_steps=50, save_strategy='steps', save_steps=50, fp16=True, report_to='none', seed=42)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds, dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, args=args)
trainer.train()

## 5. Save adapter, infer on the held-out test set, and export predictions

The exported schema (`id`, `ground_truth`, `prediction`) is consumed unchanged by `src/eval.py`.

In [ ]:
import json, torch
model.save_pretrained(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
FastLanguageModel.for_inference(model)
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as out:
    for row in ds['test']:
        prompt = build_prompt(row['code'])
        inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        out.write(json.dumps({'id': str(row['id']), 'ground_truth': row['response'], 'prediction': parse_binary_prediction(completion)}) + '\n')
print('Adapter:', OUTPUT_DIR); print('Predictions:', PREDICTIONS_FILE)